# **Loading Packages**
_____

In [12]:
import pandas as pd
import numpy as np

import requests
import time
from io import StringIO
import os

from fredapi import Fred
import yfinance as yf

# **Calling Data**
______

## **FRED**

In [14]:
os.chdir('/Users/dannyhogan/Desktop/ST-498/Code')
with open("API_key.txt", "r") as file:
    api_key = file.read().strip()
fred = Fred(api_key=api_key)

exchange_rate = fred.get_series('RBGBBIS').resample('QS').first()
oil_price = fred.get_series('DCOILBRENTEU').resample('QS').first()
bond_yield = fred.get_series('IRLTLT01GBM156N').resample('QS').first()
UK_GDP = fred.get_series('NGDPRSAXDCGBQ')
UK_Credit = fred.get_series('QGBCAMUSDA')

Fred_Data = pd.DataFrame({
    'ExchangeRate': exchange_rate,
    'OilPrice': oil_price,
    'BondYield': bond_yield,
    'rGDP': UK_GDP,
    'UK_Credit': UK_Credit.pct_change(4) * 100  
})

Fred_Data = Fred_Data[Fred_Data.index > '1993-10-01']

## **Office of National Stats**

## **Yahoo Finance**

In [15]:
ftse = yf.Ticker("^FTSE")

ftse_data = ftse.history(
    start="1993-01-01",  
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="3mo"  
)

Thoughts on alternative resampling options:
- ftse_quarterly = ftse_data.resample('QE').mean()  
- ftse_quarterly = ftse_data.resample('QE').first()  
- ftse_quarterly = ftse_data['Close'].resample('QE').last()  

## **OECD**

In [19]:
def fetch_oecd(url, name):
    """Fetch OECD data and return cleaned DataFrame"""
    if 'format=' not in url:
        url += '&format=csvfilewithlabels'
    
    headers = {
        'Accept': 'application/vnd.sdmx.data+csv; charset=utf-8; labels=both',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    time.sleep(1)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        df_clean = df[['TIME_PERIOD', 'OBS_VALUE']].copy()
        df_clean.columns = ['date', name]
        df_clean[name] = pd.to_numeric(df_clean[name], errors='coerce')
        return df_clean
    else:
        print(f"{name}: Error {response.status_code}")
        return None

# ======================================================================
# Update URLs here
# ======================================================================
queries = {
    "house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/GBR.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "cpi": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/GBR.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/GBR..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

dfs = [fetch_oecd(url, name) for name, url in queries.items()]
OECDdata = pd.concat([df.set_index('date') for df in dfs if df is not None], axis=1).reset_index()



# **Creating Dataset**
___

In [20]:
OECDdata['date'] = pd.to_datetime(OECDdata['date']).dt.tz_localize(None)
OECDdata.set_index('date', inplace=True)

Fred_Data.index = pd.to_datetime(Fred_Data.index).tz_localize(None)
ftse_data.index = pd.to_datetime(ftse_data.index).tz_localize(None)

merged = OECDdata.join([Fred_Data, ftse_data], how='outer')

/var/folders/ts/x847bxb170n3msnnpmdpjll00000gn/T/ipykernel_19605/172197089.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  OECDdata['date'] = pd.to_datetime(OECDdata['date']).dt.tz_localize(None)


In [21]:
merged = merged.drop(['High','Close','Low','Volume','Dividends','Stock Splits'],axis=1)
merged = merged[merged.index > '1993-12-31']

# **Adding Lag Variables for Regression Models:**
______

In [ ]:
os.chdir('/Users/dannyhogan/Desktop/ST-498')
df = pd.read_csv("Data/MacroVariablesUK.csv", index_col=0)

### **Variable Lag Amount**

Credit Growth: 3 Quaters (Cite)

In [25]:
from statsmodels.tsa.stattools import ccf, grangercausalitytests
from statsmodels.tsa.api import VAR

import pandas as pd
import numpy as np
from fredapi import Fred
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller


In [26]:
def check_stationarity(series, name):
    result = adfuller(series.dropna())
    pval = result[1]
    print(f"{name:15s} | ADF p-value: {pval:.4f} | {'Stationary ✓' if pval < 0.05 else 'Non-stationary ✗'}")

for col in df.columns:
    check_stationarity(df[col], col)

house_prices    | ADF p-value: 0.4047 | Non-stationary ✗
cpi             | ADF p-value: 0.0037 | Stationary ✓
unemployment    | ADF p-value: 0.1760 | Non-stationary ✗
ExchangeRate    | ADF p-value: 0.5384 | Non-stationary ✗
OilPrice        | ADF p-value: 0.1054 | Non-stationary ✗
BondYield       | ADF p-value: 0.5802 | Non-stationary ✗
rGDP            | ADF p-value: 0.6995 | Non-stationary ✗
UK_Credit       | ADF p-value: 0.1062 | Non-stationary ✗
Open            | ADF p-value: 0.7992 | Non-stationary ✗


In [28]:
# Most macro series will be non-stationary → first difference or log-diff
df_transformed = pd.DataFrame()

# Log-difference for levels (GDP, CPI, HousePrices, OilPrice, ExchangeRate)
for col in ['rGDP', 'house_prices', 'OilPrice', 'ExchangeRate','Open']:
    df_transformed[f'd_{col}'] = np.log(df[col]).diff()

for col in ['UK_Credit', 'unemployment', 'BondYield']:
    df_transformed[f'd_{col}'] = df[col].diff()

df_transformed.dropna(inplace=True)

In [36]:
def build_lag_matrix(df, target, max_lag=4):
    X = pd.DataFrame()
    for col in df.columns:
        if col == target:
            continue  # ← skip the target variable entirely
        for lag in range(1, max_lag + 1):
            X[f'{col}_L{lag}'] = df[col].shift(lag)
    
    y = df[target]
    X = X.dropna()
    y = y[y.index.isin(X.index)]
    return X, y

X, y = build_lag_matrix(df_transformed, 'd_UK_Credit', max_lag=4)

lasso = LassoCV(cv=5, max_iter=10000)
lasso.fit(X, y)

important = pd.Series(lasso.coef_, index=X.columns)
surviving = important[important != 0].sort_values()
print(surviving)

d_unemployment_L2    3.7987
dtype: float64


# **Saving Data Locally**
_____

In [ ]:
#Be sure to change to your own internal directory
os.chdir('/Users/dannyhogan/Desktop/ST-498')
merged.to_csv("Data/MacroVariablesUK.csv")
